# 🚀 TurboImage AI (Z-Image-Turbo) - Backend Seguro

**Instruções de Configuração:**
1. Vá no painel lateral esquerdo (ícone da chave 🔑 - Secrets).
2. Adicione `HF_TOKEN` com seu token do Hugging Face.
3. Adicione `NGROK_TOKEN` com seu token do ngrok.
4. Ative a permissão de acesso para ambos.

In [ ]:
# 1. Instalação das dependências
!pip install git+https://github.com/huggingface/diffusers.git -U
!pip install fastapi uvicorn pillow pyngrok torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121
!pip install python-multipart huggingface_hub

In [ ]:
# 2. Autenticação Segura
import os
from google.colab import userdata
from huggingface_hub import login
from pyngrok import ngrok

try:
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        login(token=hf_token)
        os.environ["HF_TOKEN"] = hf_token
        print("✅ Hugging Face autenticado!")
    
    ngrok_token = userdata.get('NGROK_TOKEN')
    if ngrok_token:
        ngrok.set_auth_token(ngrok_token)
        print("✅ ngrok autenticado!")
except Exception as e:
    print(f"⚠️ Verifique se os nomes HF_TOKEN e NGROK_TOKEN estão corretos nos Secrets: {e}")

In [ ]:
# 3. Backend e Carregamento do Modelo
import torch
from diffusers import ZImagePipeline, ZImageImg2ImgPipeline
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from PIL import Image
import base64
from io import BytesIO
import time
import uvicorn
import threading

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Configuração de hardware
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

print(f"⏳ Carregando modelos no {device}... Isso pode levar alguns minutos devido ao tamanho (32GB).")

pipe_t2i = ZImagePipeline.from_pretrained(
    "Tongyi-MAI/Z-Image-Turbo",
    torch_dtype=dtype,
    low_cpu_mem_usage=False
)
pipe_t2i.to(device)

pipe_i2i = ZImageImg2ImgPipeline(
    vae=pipe_t2i.vae,
    text_encoder=pipe_t2i.text_encoder,
    tokenizer=pipe_t2i.tokenizer,
    transformer=pipe_t2i.transformer,
    scheduler=pipe_t2i.scheduler,
)
pipe_i2i.to(device)

if device == "cuda":
    pipe_t2i.transformer.compile()

ASPECT_RATIOS = {
    "1:1": (1024, 1024), "16:9": (1216, 704), "9:16": (704, 1216),
    "4:3": (1152, 864), "3:4": (864, 1152), "3:2": (1216, 832), "2:3": (832, 1216)
}

class GenerateRequest(BaseModel):
    prompt: str
    negative_prompt: str = ""
    steps: int = 8
    strength: float = 0.7
    aspect_ratio: str = "1:1"
    image_base64: str = None

@app.post("/generate")
async def generate(req: GenerateRequest):
    start_time = time.time()
    width, height = ASPECT_RATIOS.get(req.aspect_ratio, (1024, 1024))
    
    try:
        if req.image_base64:
            img_data = base64.b64decode(req.image_base64.split(",")[-1])
            init_image = Image.open(BytesIO(img_data)).convert("RGB")
            init_image = init_image.resize((width, height))
            
            output = pipe_i2i(
                prompt=req.prompt,
                negative_prompt=req.negative_prompt,
                image=init_image,
                strength=req.strength,
                num_inference_steps=req.steps,
                guidance_scale=0.0,
            ).images[0]
        else:
            output = pipe_t2i(
                prompt=req.prompt,
                negative_prompt=req.negative_prompt,
                width=width,
                height=height,
                num_inference_steps=req.steps,
                guidance_scale=0.0,
            ).images[0]
            
        buffered = BytesIO()
        output.save(buffered, format="JPEG", quality=90)
        img_str = base64.b64encode(buffered.getvalue()).decode()
        
        return {
            "image": f"data:image/jpeg;base64,{img_str}",
            "latency": f"{time.time() - start_time:.2f}s"
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

def start_api():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=start_api, daemon=True).start()
time.sleep(5)
public_url = ngrok.connect(8000).public_url
print(f"\n🚀 BACKEND ONLINE: {public_url}")